# Segment 2 — LiDAR Processing & Visualization

Loads a KITTI/SemanticKITTI-style `.bin` LiDAR frame, cleans it, applies distance/ROI filters, separates ground/non-ground points, and visualizes the result with Open3D.

In [17]:
# Install dependencies if needed
!pip install numpy open3d -q


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os
import numpy as np
import open3d as o3d

In [19]:
# Configuration
FILE_PATH = "fake_lidar_000000.bin"
MAX_DISTANCE = 100.0
GROUND_HEIGHT = 0.2

print("LiDAR file:", FILE_PATH)

LiDAR file: fake_lidar_000000.bin


In [20]:
def load_lidar(file_path):
    """Load a KITTI/SemanticKITTI-style .bin file: x, y, z, intensity."""
    points = np.fromfile(file_path, dtype=np.float32)
    if len(points) % 4 != 0:
        raise ValueError("Invalid LiDAR file: values are not divisible by 4.")
    points = points.reshape(-1, 4)
    return points[:, :3], points[:, 3]


def remove_invalid_points(xyz, intensity):
    mask = np.isfinite(xyz).all(axis=1) & np.isfinite(intensity)
    return xyz[mask], intensity[mask]


def filter_distance(xyz, intensity, max_distance=100.0):
    distance = np.linalg.norm(xyz, axis=1)
    mask = distance <= max_distance
    return xyz[mask], intensity[mask]


def filter_roi(xyz, intensity, x_min=-10, x_max=100,
               y_min=-50, y_max=50, z_min=-3, z_max=5):
    mask = (
        (xyz[:, 0] >= x_min) & (xyz[:, 0] <= x_max) &
        (xyz[:, 1] >= y_min) & (xyz[:, 1] <= y_max) &
        (xyz[:, 2] >= z_min) & (xyz[:, 2] <= z_max)
    )
    return xyz[mask], intensity[mask]


def separate_ground(xyz, intensity, ground_height=0.2):
    ground_mask = xyz[:, 2] < ground_height
    return (
        xyz[ground_mask], intensity[ground_mask],
        xyz[~ground_mask], intensity[~ground_mask]
    )


def voxel_downsample(xyz, voxel_size=0.10):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    downsampled = pcd.voxel_down_sample(voxel_size)
    return np.asarray(downsampled.points)


def create_cloud(xyz):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(xyz)
    return pcd


def visualize_ground_and_objects(ground_xyz, object_xyz):
    ground = create_cloud(ground_xyz)
    objects = create_cloud(object_xyz)
   
    ground.paint_uniform_color([0.3, 0.8, 0.3])
    objects.paint_uniform_color([0.9, 0.2, 0.2])
    o3d.visualization.draw_geometries(
        [ground, objects], window_name="LiDAR Point Cloud"
    )

In [21]:
# Run the complete processing pipeline
if not os.path.exists(FILE_PATH):
    print(f"File not found: {FILE_PATH}")
    print("Upload/create the data/000000.bin file, then run this cell again.")
else:
    print("=" * 45)
    print("        LiDAR PROCESSOR")
    print("=" * 45)

    xyz, intensity = load_lidar(FILE_PATH)
    print("Raw points:", len(xyz))

    xyz, intensity = remove_invalid_points(xyz, intensity)
    print("After invalid filtering:", len(xyz))

    xyz, intensity = filter_distance(xyz, intensity, MAX_DISTANCE)
    print("After distance filtering:", len(xyz))

    xyz, intensity = filter_roi(xyz, intensity)
    print("After ROI filtering:", len(xyz))

    ground_xyz, ground_intensity, object_xyz, object_intensity = separate_ground(
        xyz, intensity, GROUND_HEIGHT
    )

    print("\nGround points:", len(ground_xyz))
    print("Non-ground points:", len(object_xyz))
   
    print("\nOpening 3D viewer...")
    visualize_ground_and_objects(ground_xyz, object_xyz)
    print("Processing complete.")

        LiDAR PROCESSOR
Raw points: 71000
After invalid filtering: 71000
After distance filtering: 69167
After ROI filtering: 65631

Ground points: 45872
Non-ground points: 19759

Opening 3D viewer...
Processing complete.


## Notes

- The `z < 0.2` ground separator is only a prototype and will not robustly handle slopes or uneven terrain.
- The next milestone is replacing it with a better ground segmentation method and then moving to object detection/segmentation.